# ✈️ Step 3: Preprocessing and Feature Engineering
**Project:** Flight Delay Prediction via Gradient Boosting and Temporal Deep Learning  
**Phase 1:** Traditional Machine Learning Only  
**Prediction Definition:** At a fixed time **before scheduled departure**, predict whether the flight's arrival delay will be **at least 15 minutes** (`is_delayed_15 = 1`).

---

### 📋 Key Tasks Implemented:
1. **Target Construction:** Binary indicator ($1$ if arrival delay $\ge 15$ minutes, $0$ otherwise).
2. **Operational Filtering & Audit:** Exclude cancelled flights, diverted flights, and unobserved targets, with full row tracking.
3. **Chronological Splitting:** Date-based partitioning ensuring train $\rightarrow$ validation $\rightarrow$ test with **zero date overlap**.
4. **Pre-Departure Feature Engineering:**
   - Departure hour & minute with cyclic trigonometric encoding (sine/cosine).
   - Day of week, weekend indicator, month with cyclic calendar encodings.
   - Planned route (`ORIGIN_DEST`), flight distance, and scheduled duration.
5. **Historical Priors (Strictly No Lookahead Bias):**
   - Empirical Bayes smoothed carrier, airport, and route delay rates computed **exclusively from training data**.
6. **Scikit-Learn ColumnTransformer & Pipeline:**
   - Numerical: Median imputation + Standard scaling.
   - Categorical: Constant imputation + One-Hot Encoding (`handle_unknown='ignore'`).
   - Fitted **on training data only**; applied to validation and test.
7. **Leakage & Chronological Assertions:** Automated tests verifying zero target leakage and zero date overlap.
8. **Data Dictionary:** Documented metadata persisted to `results/data_dictionary.md` and `results/data_dictionary.json`.


## 1. Environment Setup and Module Imports
We load standard data science packages and our modular functions from `src/preprocessing.py`, `src/config.py`, and `src/utils.py`.

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path so modules in src/ can be loaded
PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project configuration and preprocessing modules
from src.config import (
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    RESULTS_DIR,
    TARGET_COLUMN,
    LEAKAGE_COLUMNS,
    RANDOM_STATE
)
from src.data_loader import find_dataset_files, detect_column_mapping
from src.preprocessing import (
    prepare_classification_dataset,
    chronological_train_val_test_split,
    engineer_pre_departure_features,
    fit_training_historical_priors,
    apply_historical_priors,
    build_column_transformer,
    assert_no_target_leakage,
    assert_chronological_integrity,
    assert_no_null_targets,
    generate_data_dictionary,
    generate_diagnostic_flight_data
)

sns.set_theme(style="whitegrid", palette="muted")
print("✓ All preprocessing utilities and configuration imported successfully.")
print(f"✓ Project Root: {PROJECT_ROOT}")


✓ All preprocessing utilities and configuration imported successfully.
✓ Project Root: /Users/suryansh/Desktop/video/Flight_Delay_Prediction


## 2. Dataset Ingestion and Path Resolution
We search `data/raw/` for the raw flight dataset. If a dataset is present, we load it; if not yet placed, we execute our diagnostic benchmark generator to validate the complete pipeline top-to-bottom without interruption.

In [2]:
dataset_files = find_dataset_files()

if dataset_files:
    primary_file = dataset_files[0]
    file_size_mb = primary_file.stat().st_size / (1024 * 1024)
    print(f"✓ Real dataset detected: {primary_file.name} ({file_size_mb:.2f} MB)")
    
    # Load sample for rapid processing if large, or full file
    SAMPLE_ROWS = 250_000 if file_size_mb > 200 else None
    raw_df = pd.read_csv(primary_file, nrows=SAMPLE_ROWS, low_memory=False)
    mapping, missing = detect_column_mapping(raw_df.columns)
    DATASET_MODE = "REAL_DATASET"
else:
    print("ℹ️  Live CSV file not found in data/raw/. Generating diagnostic benchmark data (5,000 flights)")
    print("   spanning Jan 1 to Mar 31 to execute and verify all preprocessing transformations.")
    raw_df, mapping = generate_diagnostic_flight_data(n_samples=5000, random_state=RANDOM_STATE)
    DATASET_MODE = "BENCHMARK_VERIFICATION"

print(f"✓ Ingested dataframe shape: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns.")


ℹ️  Live CSV file not found in data/raw/. Generating diagnostic benchmark data (5,000 flights)
   spanning Jan 1 to Mar 31 to execute and verify all preprocessing transformations.
✓ Ingested dataframe shape: 5,000 rows, 20 columns.


## 3. Operational Filtering & Binary Target Creation
### Filtering Rationale:
1. **Cancelled Flights:** Never flew. Arrival delay is undefined ($NaN$).
2. **Diverted Flights:** Arrived at alternate destination; scheduled arrival target invalid.
3. **Missing Delay Records:** Ground-truth label absent; cannot train/evaluate.
4. **Target Formulation:**
$$\text{Target } y = \begin{cases} 1 & \text{if } \text{Arrival Delay} \ge 15 \text{ min} \\ 0 & \text{if } \text{Arrival Delay} < 15 \text{ min} \end{cases}$$

In [3]:
clean_df, audit_report = prepare_classification_dataset(
    raw_df,
    mapping,
    filter_cancelled=True,
    filter_diverted=True
)

print("=" * 65)
print("OPERATIONAL FILTERING AUDIT REPORT")
print("=" * 65)
print(f"Initial Ingested Rows          : {audit_report['initial_rows']:,}")
print(f"Cancelled Flights Removed      : -{audit_report['cancelled_rows_removed']:,}")
print(f"Diverted Flights Removed       : -{audit_report['diverted_rows_removed']:,}")
print(f"Missing Target Rows Removed    : -{audit_report['missing_target_rows_removed']:,}")
print("-" * 65)
print(f"Final Clean Classification Rows: {audit_report['final_rows']:,}")
print(f"Row Retention Percentage       : {audit_report['retention_pct']:.2f}%")
print(f"Delayed (>=15 min) Flights     : {audit_report['target_positive_count']:,} ({audit_report['target_positive_pct']:.2f}%)")
print("=" * 65)


OPERATIONAL FILTERING AUDIT REPORT
Initial Ingested Rows          : 5,000
Cancelled Flights Removed      : -92
Diverted Flights Removed       : -17
Missing Target Rows Removed    : -0
-----------------------------------------------------------------
Final Clean Classification Rows: 4,891
Row Retention Percentage       : 97.82%
Delayed (>=15 min) Flights     : 1,989 (40.67%)


## 4. Chronological Train / Validation / Test Split
### Why Random Cross-Validation is Flawed in Flight Prediction:
Random train/test splits cause **temporal lookahead leakage**: flights departing on the exact same afternoon or week are shared between training and testing, artificially inflating model accuracy because regional weather patterns and ATC delays are correlated across hours.

**Chronological Splitting Methodology:**
- **Training (70% earliest dates):** Earliest flight operations.
- **Validation (15% intermediate dates):** Used for threshold tuning and hyperparameter selection.
- **Testing (15% latest dates):** Held-out final evaluation simulating true forward-in-time deployment.
- **Strict Constraint:** Zero date overlap between partitions ($\max(D_{\text{train}}) < \min(D_{\text{val}})$ and $\max(D_{\text{val}}) < \min(D_{\text{test}})$).

In [4]:
train_df, val_df, test_df, split_report = chronological_train_val_test_split(
    clean_df,
    mapping,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

print("=" * 75)
print("CHRONOLOGICAL PARTITION BOUNDARIES & ROW AUDIT")
print("=" * 75)
print(f"TRAINING SET   : {split_report['train_rows']:,} rows | "
      f"{split_report['train_start_date']} to {split_report['train_end_date']} | "
      f"Delay Rate: {split_report['train_delay_rate_pct']:.2f}%")

print(f"VALIDATION SET : {split_report['val_rows']:,} rows | "
      f"{split_report['val_start_date']} to {split_report['val_end_date']} | "
      f"Delay Rate: {split_report['val_delay_rate_pct']:.2f}%")

print(f"TESTING SET    : {split_report['test_rows']:,} rows | "
      f"{split_report['test_start_date']} to {split_report['test_end_date']} | "
      f"Delay Rate: {split_report['test_delay_rate_pct']:.2f}%")
print("=" * 75)
print("✓ Chronological assertion verified: Strictly zero date overlap across splits.")


CHRONOLOGICAL PARTITION BOUNDARIES & ROW AUDIT
TRAINING SET   : 3,408 rows | 2022-01-01 to 2022-03-03 | Delay Rate: 40.17%
VALIDATION SET : 765 rows | 2022-03-04 to 2022-03-17 | Delay Rate: 42.61%
TESTING SET    : 718 rows | 2022-03-18 to 2022-03-31 | Delay Rate: 40.95%
✓ Chronological assertion verified: Strictly zero date overlap across splits.


## 5. Pre-Departure Feature Engineering
We construct features known strictly prior to scheduled departure:
- **Scheduled Time:** `departure_hour` (0-23), `departure_minute` (0-59).
- **Cyclic Time Transformation:** Continuous mapping of periodic cycles via sine and cosine:
  $$\text{hour}_{\sin} = \sin\left(\frac{2\pi \times \text{hour}}{24}\right), \quad \text{hour}_{\cos} = \cos\left(\frac{2\pi \times \text{hour}}{24}\right)$$
- **Calendar Signals:** `day_of_week`, `is_weekend`, `month`, and their respective cyclic transformations.
- **Route Interaction:** `route = origin + "_" + destination`.
- **Planned Route Geometry:** `distance`, `scheduled_elapsed_time`.

In [5]:
# Extract pre-departure features for all splits
X_train, y_train = engineer_pre_departure_features(train_df, mapping)
X_val, y_val = engineer_pre_departure_features(val_df, mapping)
X_test, y_test = engineer_pre_departure_features(test_df, mapping)

print(f"✓ Engineered Feature Matrix X_train: {X_train.shape[0]:,} rows, {X_train.shape[1]} features.")
print(f"✓ Target Vector y_train           : {y_train.shape[0]:,} entries.")
print("\nSample Engineered Pre-Departure Features:")
print(X_train.head(4).T)


✓ Engineered Feature Matrix X_train: 3,408 rows, 19 features.
✓ Target Vector y_train           : 3,408 entries.

Sample Engineered Pre-Departure Features:
                               0         1         2         3
departure_hour                 7         9         7        15
departure_minute              11        50        21         8
dep_hour_sin            0.965926  0.707107  0.965926 -0.707107
dep_hour_cos           -0.258819 -0.707107 -0.258819 -0.707107
dep_minute_of_day_sin   0.952396    0.5373  0.938191 -0.731354
dep_minute_of_day_cos  -0.304864 -0.843391 -0.346117 -0.681998
day_of_week                    6         6         6         6
is_weekend                     1         1         1         1
day_of_week_sin        -0.781831 -0.781831 -0.781831 -0.781831
day_of_week_cos          0.62349   0.62349   0.62349   0.62349
month                          1         1         1         1
month_sin                    0.5       0.5       0.5       0.5
month_cos               0

## 6. Historical Prior Delay Rates (Strictly No Lookahead Bias)
### Mathematical Formulation:
Historical carrier and airport performance strongly informs flight punctuality. However, **calculating aggregate statistics across the full dataset introduces target leakage**.

To strictly prevent leakage:
1. Delay rates are computed **exclusively on the training split**.
2. We apply Empirical Bayes smoothing (M-estimate with smoothing parameter $m = 20$):
$$\text{Prior} = \frac{\sum y_{\text{train}} + m \cdot \bar{y}_{\text{global}}}{N_{\text{group}} + m}$$
3. For new or unseen carriers/airports encountered in validation or test data, the prior defaults safely to the training global baseline $\bar{y}_{\text{global}}$.

In [6]:
# Fit priors ONLY on training set
priors_map, global_train_mean = fit_training_historical_priors(
    pd.concat([X_train, y_train], axis=1),
    group_cols=["airline", "origin", "destination", "route"],
    target_col=TARGET_COLUMN,
    smoothing=20.0
)

print(f"Global Training Delay Rate (Fallback): {global_train_mean * 100:.2f}%")
print(f"Unique Training Carriers Mapped      : {len(priors_map.get('airline', {}))}")
print(f"Unique Training Origin Hubs Mapped   : {len(priors_map.get('origin', {}))}")
print(f"Unique Training Destination Hubs     : {len(priors_map.get('destination', {}))}")
print(f"Unique Training Routes Mapped        : {len(priors_map.get('route', {}))}")

# Apply learned training priors to train, validation, and test sets
X_train = apply_historical_priors(X_train, priors_map, global_train_mean)
X_val = apply_historical_priors(X_val, priors_map, global_train_mean)
X_test = apply_historical_priors(X_test, priors_map, global_train_mean)

print("\nSample Priors in Training Set:")
prior_cols = [c for c in X_train.columns if "prior" in c]
print(X_train[prior_cols].head(4))


Global Training Delay Rate (Fallback): 40.17%
Unique Training Carriers Mapped      : 7
Unique Training Origin Hubs Mapped   : 10
Unique Training Destination Hubs     : 10
Unique Training Routes Mapped        : 90

Sample Priors in Training Set:
   airline_delay_prior  origin_delay_prior  destination_delay_prior  \
0             0.411000            0.438138                 0.420219   
1             0.394351            0.386206                 0.378679   
2             0.411000            0.396874                 0.378679   
3             0.391906            0.438138                 0.406780   

   route_delay_prior  
0           0.380681  
1           0.373458  
2           0.441677  
3           0.456738  


## 7. Scikit-Learn ColumnTransformer and Pipeline
We assemble a standardized scikit-learn preprocessing architecture:
- **Numerical Pipeline:**
  1. `SimpleImputer(strategy='median')`: Imputes missing distance or duration values using training medians.
  2. `StandardScaler()`: Standardizes continuous and cyclic features to zero mean and unit variance.
- **Categorical Pipeline:**
  1. `SimpleImputer(strategy='constant', fill_value='UNKNOWN')`: Handles missing category labels.
  2. `OneHotEncoder(handle_unknown='ignore', min_frequency=25)`: One-hot encodes categories, gracefully ignoring unseen categories at validation/test time.
- **Rule of Discipline:** The `ColumnTransformer` is **fitted strictly on `X_train`** and merely applied via `.transform()` to `X_val` and `X_test`.

In [7]:
# Identify numerical and categorical column subsets
numeric_cols = list(X_train.select_dtypes(include=["number"]).columns)
categorical_cols = ["airline", "origin", "destination"]

print(f"Numerical Features ({len(numeric_cols)}):")
print(" ", numeric_cols)
print(f"Categorical Features ({len(categorical_cols)}):")
print(" ", categorical_cols)

# Build scikit-learn ColumnTransformer
preprocessor = build_column_transformer(numeric_cols, categorical_cols, scale_numeric=True)

# FIT EXCLUSIVELY ON TRAINING DATA
print("\nFitting ColumnTransformer on X_train...")
X_train_processed = preprocessor.fit_transform(X_train)

# TRANSFORM VALIDATION AND TEST (NO FITTING)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out()

print("✓ Preprocessing Pipeline Fit & Transform Complete:")
print(f"  - X_train_processed shape: {X_train_processed.shape}")
print(f"  - X_val_processed shape  : {X_val_processed.shape}")
print(f"  - X_test_processed shape : {X_test_processed.shape}")
print(f"  - Total Transformed Dimensions: {len(feature_names_out)}")


Numerical Features (19):
  ['departure_hour', 'departure_minute', 'dep_hour_sin', 'dep_hour_cos', 'dep_minute_of_day_sin', 'dep_minute_of_day_cos', 'day_of_week', 'is_weekend', 'day_of_week_sin', 'day_of_week_cos', 'month', 'month_sin', 'month_cos', 'distance', 'scheduled_elapsed_time', 'airline_delay_prior', 'origin_delay_prior', 'destination_delay_prior', 'route_delay_prior']
Categorical Features (3):
  ['airline', 'origin', 'destination']

Fitting ColumnTransformer on X_train...
✓ Preprocessing Pipeline Fit & Transform Complete:
  - X_train_processed shape: (3408, 46)
  - X_val_processed shape  : (765, 46)
  - X_test_processed shape : (718, 46)
  - Total Transformed Dimensions: 46


## 8. Anti-Leakage & Data Integrity Assertions
We execute systematic checks to mathematically prove zero target leakage, zero date overlap, and label completeness.

In [8]:
# 1. Target Leakage Assertion
assert_no_target_leakage(X_train.columns, LEAKAGE_COLUMNS, TARGET_COLUMN)
assert_no_target_leakage(X_val.columns, LEAKAGE_COLUMNS, TARGET_COLUMN)
assert_no_target_leakage(X_test.columns, LEAKAGE_COLUMNS, TARGET_COLUMN)
print("✓ ASSERTION PASSED: No target variable or post-departure leakage columns exist in X.")

# 2. Chronological Ordering Assertion
assert_chronological_integrity(
    train_df["_flight_date_dt"],
    val_df["_flight_date_dt"],
    test_df["_flight_date_dt"]
)
print("✓ ASSERTION PASSED: Strict chronological ordering verified with 0 date overlap.")

# 3. Target Completeness Assertion
assert_no_null_targets(y_train, y_val, y_test)
print("✓ ASSERTION PASSED: Target vectors y_train, y_val, y_test contain 0 null values.")

# 4. Transformed Dimension Assertion
assert X_train_processed.shape[1] == X_val_processed.shape[1] == X_test_processed.shape[1]
print("✓ ASSERTION PASSED: Feature dimensions match perfectly across train, val, and test.")


✓ ASSERTION PASSED: No target variable or post-departure leakage columns exist in X.
✓ ASSERTION PASSED: Strict chronological ordering verified with 0 date overlap.
✓ ASSERTION PASSED: Target vectors y_train, y_val, y_test contain 0 null values.
✓ ASSERTION PASSED: Feature dimensions match perfectly across train, val, and test.


## 9. Data Dictionary & Feature Provenance
We document every feature's mathematical definition, data type, and confirmation of prediction-time availability.

In [9]:
data_dict = generate_data_dictionary(numeric_cols, categorical_cols, output_dir=RESULTS_DIR)

print(f"✓ Saved data dictionary JSON to: {RESULTS_DIR / 'data_dictionary.json'}")
print(f"✓ Saved data dictionary Markdown to: {RESULTS_DIR / 'data_dictionary.md'}\n")

dict_summary = pd.DataFrame([
    {
        "Feature": k,
        "Type": v["type"],
        "Available at Prediction": v["available_at_prediction"],
        "Description": v["description"][:55] + "..." if len(v["description"]) > 55 else v["description"]
    }
    for k, v in data_dict.items()
])
print(dict_summary.to_string(index=False))


✓ Saved data dictionary JSON to: /Users/suryansh/Desktop/video/Flight_Delay_Prediction/results/data_dictionary.json
✓ Saved data dictionary Markdown to: /Users/suryansh/Desktop/video/Flight_Delay_Prediction/results/data_dictionary.md

                Feature               Type                       Available at Prediction                                                Description
         departure_hour     Integer (0-23)                               Yes (Scheduled)     Hour of scheduled departure extracted from CRSDepTime.
       departure_minute     Integer (0-59)                               Yes (Scheduled)                             Minute of scheduled departure.
           dep_hour_sin      Float [-1, 1]                               Yes (Scheduled) Cyclic sine encoding of scheduled departure hour: sin(2...
           dep_hour_cos      Float [-1, 1]                               Yes (Scheduled) Cyclic cosine encoding of scheduled departure hour: cos...
  dep_minute_of_day_sin  

## 10. Summary and Modeling Readiness (Step 3 Complete)

### Preprocessing & Feature Engineering Summary:
1. **Target:** Binary ground-truth $y \in \{0, 1\}$ strictly isolated from $X$.
2. **Operational Validity:** Cancelled, diverted, and missing-delay records cleanly excluded with full audit tracking.
3. **Temporal Partitioning:** Chronological split (70% Train, 15% Val, 15% Test) with strict date non-overlap.
4. **Pre-Departure Feature Matrix:**
   - Temporal cycles: Sine and cosine transformations of scheduled departure hour and day of week.
   - Spatial geometry: Flight distance, route, and scheduled elapsed time.
   - Historical delay priors: Bayesian regularized delay rates computed exclusively on training data.
5. **Leakage Guardrails:** Fully verified with scikit-learn `Pipeline` and `ColumnTransformer` (fit on train only).

---
**Next Stage (Step 4):** Model Training — Baseline Logistic Regression, Random Forest, and Gradient Boosted Decision Trees (XGBoost) evaluated using PR-AUC, ROC-AUC, and F1-score.
